In [ ]:
import gmsh
import math
import os
import sys
import time


path_from = '../../s2.0.1/s2.0.1/breps/step'
# path_save = '../../s2.0.1/s2.0.1/MSH_volumes_with_surfaces'
path_save = ''
i = 0
e = 0
for filename in os.listdir(path_from):
    print(filename)
    # gmsh.initialize()

    # gmsh.model.add("t20")

    # Load a STEP file (using `importShapes' instead of `merge' allows to directly
    # retrieve the tags of the highest dimensional imported entities):
    try:
        gmsh.initialize()

        gmsh.model.add(filename)

        # Load a STEP file (using `importShapes' instead of `merge' allows to directly
        # retrieve the tags of the highest dimensional imported entities):

        v = gmsh.model.occ.importShapes(os.path.join(path_from,filename))
        # v = gmsh.model.occ.Merge(os.path.join(path_from,filename))

        gmsh.model.occ.synchronize()
        # print(gmsh.model.getEntities(2))
        # gmsh.model.removeEntities(gmsh.model.getEntities(2))
        # print('aft',gmsh.model.getEntities(2))
        # gmsh.model.removeEntities(gmsh.model.getEntities(1))
        # gmsh.model.removeEntities(gmsh.model.getEntities(0))


        # Finally, let's specify a global mesh size and mesh the partitioned model:
        gmsh.option.setNumber("Mesh.MeshSizeMin", 0.5)
        gmsh.option.setNumber("Mesh.MeshSizeMax", 1)
        # gmsh.option.setNumber("Mesh.MeshSizeMin", 1)
        # gmsh.option.setNumber("Mesh.MeshSizeMax", 1)
        # gmsh.option.setNumber("Mesh.CharacteristicLengthMin", 10)
        # gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 10)

        surfaces = gmsh.model.getEntities(dim=2)
        # print(len(surfaces))
        gmsh.model.addPhysicalGroup(2, [surfaces[i][1] for i in range(len(surfaces))],tag = 12)
        # gmsh.model.addPhysicalGroup(2, [surfaces[i][1]], 12)

        volumes = gmsh.model.getEntities(dim=3)
        # print(volumes)
        gmsh.model.addPhysicalGroup(volumes[0][0], [volumes[0][1]], 11)
        gmsh.model.setPhysicalName(volumes[0][0], 11, "Part volume")


        # gmsh.model.setPhysicalName(surfaces[0][0], 11, "Part surface")
        gmsh.model.occ.synchronize()
        gmsh.model.mesh.generate(3)
        # gmsh.write(filename[:-4]+'.msh')
        gmsh.write(os.path.join(path_save,filename[:-4] + ".msh"))

        # Launch the GUI to see the results:
        if '-nopopup' not in sys.argv:
            gmsh.fltk.run()

        gmsh.finalize()
        
        i += 1
        
        if i>0:
            break
        if i%100 == 0:
            print('i: ',i)
    except:
        e += 1
        if e%100 == 0:
            print('e: ',e)
        # print('error in: ',filename)
        continue

In [ ]:
import meshio
import numpy as np
from matplotlib.pyplot import plt

def get_midpt_and_range(lims):
    midpt = (lims[1] + lims[0])/2.
    span  = abs(lims[1] - lims[0])
    return midpt, span

def equal_axes(ax):
    x_m, x_r = get_midpt_and_range(ax.get_xlim3d())
    y_m, y_r = get_midpt_and_range(ax.get_ylim3d())
    z_m, z_r = get_midpt_and_range(ax.get_zlim3d())

    r = max([x_r, y_r, z_r])/2.
    ax.set_xlim3d([x_m - r, x_m + r])
    ax.set_ylim3d([y_m - r, y_m + r])
    ax.set_zlim3d([z_m - r, z_m + r])
def plot_fields(xyz, fields, titles = None, cmap="coolwarm"):
    if titles is not None:
        assert len(fields) == len(titles)
    N = len(fields)

    # vmin = min([min(f) for f in fields])
    # vmax = max([max(f) for f in fields])

    fig = plt.figure(figsize=(3.5*N,4),dpi=300)
    for i, field in enumerate(fields):
        ax = fig.add_subplot(1, N, i+1, projection='3d')
        scatter = ax.scatter(xyz[:,0],xyz[:,1],xyz[:,2],c=field, cmap=cmap)#, vmin = vmin, vmax = vmax)
        # ax.set_xticklabels([])
        # ax.set_yticklabels([])
        # ax.set_zticklabels([])
        equal_axes(ax)
        plt.title(titles[i])
        cbar = fig.colorbar(scatter, format="% .2f", shrink=0.8)

    plt.show()


msh = meshio.read(os.path.join('100028_0c3a8f1c_18.msh'))

# print(list(set(msh.cells[6].data.reshape(-1))))
print((msh.cells))
# print(msh.get_cell_data("gmsh:physical"))
ind = []
for i in range(len(msh.cells)-1):
    ind.extend(list(set(msh.cells[i].data.reshape(-1))))
# print([ind.extend(list(set(msh.cells[i].data.reshape(-1)))) for i in range(len(msh.cells)-1)])
# total_ind = [ind.extend(list(set(msh.cells[i].data.reshape(-1)))) for i in range(len(msh.cells)-1)]

verts = msh.points[ind]
vertsh = verts[np.where(verts[:,2]<0)]
vertsh = vertsh[np.where(vertsh[:,0]<0)]
gt = np.zeros_like(vertsh)
# print(set(msh.cells[0].data.reshape(-1)))
print(np.shape(verts))
# print(gt)
plot_fields(vertsh, [gt], ["Ground Truth"], cmap="jet")